# Data Cleaning : l'agent IA

Ce notebook prolonge [`01_manual_cleaning.ipynb`](./01_manual_cleaning.ipynb) : nous passons du nettoyage manuel à un workflow agentique avec validation humaine.

Le projet utilise `dirty_sales_data.csv`, un jeu de transactions contenant notamment les colonnes `Transaction_ID`, `Date`, `Client_Name`, `Product_Category`, `Unit_Price` et `Quantity`.

**Workflow :**
1. **Analyse** : l'IA examine les données et propose des actions.
2. **Validation** : l'utilisateur choisit les actions pertinentes (simulé ici).
3. **Exécution** : le code de nettoyage est généré puis appliqué au DataFrame.

Les chemins du notebook sont relatifs à la racine du projet afin de pouvoir partager le dépôt sans dépendre d'un chemin propre à un ordinateur. Le jeu de données est transmis au service Gemini : vérifiez qu'il ne contient pas d'informations confidentielles avant d'exécuter les cellules.

## 1. Installation & configuration

Depuis la racine du projet, installez les dépendances avec `python -m pip install -r requirements.txt`. Créez ensuite `.env` à partir de `.env.example` et renseignez votre clé `GEMINI_API_KEY`. Ne partagez jamais `.env` : il sera exclu du dépôt Git.

In [ ]:
import pandas as pd
from google import genai
import os
import json
import io
from pathlib import Path
from dotenv import load_dotenv
import tabulate

# Nouveauté : Pydantic pour structurer la sortie
from pydantic import BaseModel, Field
from typing import List

# Les notebooks doivent être lancés depuis la racine du projet.
PROJECT_ROOT = Path.cwd()
data_path = PROJECT_ROOT / 'dirty_sales_data.csv'
if not data_path.is_file():
    raise FileNotFoundError(
        f"{data_path} introuvable. Ouvrez le notebook depuis la racine du projet."
    )

# 1. Chargement de la clé depuis le fichier local .env
load_dotenv(PROJECT_ROOT / '.env')
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError(
        "GEMINI_API_KEY manquante. Configurez-la dans le fichier .env."
    )

# 2. Config Gemini
MODEL_NAME = 'gemini-3.6-flash'
client = genai.Client(api_key=api_key)

def generate_content(prompt, response_schema=None):
    config = None
    if response_schema is not None:
        config = types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=response_schema,
        )
    return client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=config,
    )

# 3. Chargement Data
df = pd.read_csv(data_path)
print("Données chargées :", df.shape)

In [ ]:
df.info()

In [ ]:
#buffer va permetter de transformer le .info en une string exploitable
buffer = io.StringIO()
df.info(buf=buffer)
info_str = buffer.getvalue()
print(info_str)
head_str = df.head(20).to_markdown()
print(head_str)

## 2. Phase 1 : Analyse Automatique

Nous avons deux façons de demander à l'IA :

### Approche V1 : La méthode naïve (Texte libre)
On demande simplement : "Analyse ça".

In [ ]:
prompt_v1 = f"""
Tu es un Data Analyst. Analyse ce dataset (info et head) et dis-moi ce qui ne va pas.
Sois concis.

DATA INFO:
{info_str}

DATA HEAD (Markdown):
{head_str}
"""

print(prompt_v1)

response_v1 = generate_content(prompt_v1)
print(response_v1.text)

In [ ]:
print(response_v1.text)

 **Problème** : C'est très lisible pour un humain, mais **impossible à utiliser dans une application** (comment créer des checkbox automatiquement à partir de ce texte ?).

### Approche V2 : La méthode industrielle (Structured Output)
On va définir un **Schéma de données** rigoureux avec `Pydantic` et forcer l'IA à le respecter.
C'est la méthode "Structured Output".

In [ ]:
# Définition de la structure attendue
class DataIssue(BaseModel):
    column: str = Field(..., description="Nom de la colonne concernée")
    issue_type: str = Field(..., description="Type de problème (ex: 'Format', 'Duplicata', 'Valeur manquante')")
    description: str = Field(..., description="Explication courte du problème")
    suggested_action: str = Field(..., description="Action Python suggérée (ex: 'pd.to_datetime', 'drop_duplicates')")

class DataAudit(BaseModel):
    issues: List[DataIssue]

# Appel à Gemini avec le schéma
prompt_v2 = f"""
Audite ce jeu de données de ventes et détecte les anomalies confirmées de types, doublons et formats.
Tiens compte de ces colonnes : Transaction_ID (identifiant), Date, Client_Name, Product_Category, Unit_Price (prix unitaire) et Quantity (quantité).
Signale les dates non reconnues, les prix non numériques, les identifiants répétés et les variantes manifestes de catégories. Ne déduis pas de règle métier ni de correction qui ne soit pas justifiée par les données. Remplis la structure JSON demandée.

DATA INFO:
{info_str}

DATA HEAD:
{head_str}
"""

response_v2 = generate_content(prompt_v2, response_schema=DataAudit)

# Parsing automatique
audit_result = json.loads(response_v2.text)
print(audit_result)

In [ ]:
issues_list = audit_result['issues']
# Affichage utilisable par le code
pd.DataFrame(issues_list)

 **Succès** : Nous avons maintenant une liste d'objets structurés que notre interface pourra manipuler.

## 3. Phase 2 : Simulation "Human-in-the-loop"

Dans l'app finale, l'utilisateur validera ces suggestions.
Ici, on garde tout.

In [ ]:
selected_actions = issues_list
print(f"Validation simulée de {len(selected_actions)} actions.")

selected_actions_restricted = selected_actions[0:3]
print(f"Validation simulée de {len(selected_actions_restricted)} actions.")

In [ ]:
selected_actions
selected_actions_restricted

## 4. Phase 3 : Génération, vérification & exécution du code

Dernière étape : transformer ces instructions JSON en code Python. Le code retourné par un modèle n'est pas sûr par défaut : lisez-le et vérifiez-le avant de l'exécuter. `exec` exécute du code arbitraire et ne constitue pas un bac à sable.

In [ ]:
# Définition de la structure de sortie pour le code
class CodeOutput(BaseModel):
    python_script: str = Field(..., description="Le script Python complet et exécutable, sans blocs markdown.")

def generate_cleaning_code(df_sample, actions):
    cols_context = df_sample.dtypes.to_dict()
    
    prompt_code = f"""
    Tu es un développeur Python Senior spécialisé dans le nettoyage de données de ventes avec Pandas.
    Génère le code Python pour nettoyer le dataframe 'df' selon ces actions validées :
    {json.dumps(actions, indent=2)}
    df : {df_sample.head(10)}
    Colonnes : {cols_context}
    
    Règles :
    1. Le jeu décrit des ventes : Transaction_ID, Date, Client_Name, Product_Category, Unit_Price et Quantity.
    2. Le code agit directement sur 'df', sans supprimer de lignes ni inventer de valeurs sans action validée.
    3. Gère les conversions avec robustesse et conserve les valeurs non convertibles comme valeurs manquantes.
    """
    
    # On force une sortie structurée contenant le code
    response = generate_content(prompt_code, response_schema=CodeOutput)
    
    # On récupère proprement le script dans le JSON
    return json.loads(response.text)['python_script']

cleaning_code = generate_cleaning_code(df, selected_actions)
print(cleaning_code)

In [ ]:
cleaning_code_restricted = generate_cleaning_code(df, selected_actions_restricted)
print(cleaning_code_restricted)

In [ ]:
# Après vérification manuelle du script généré : exécution dans un espace de noms local.
df_clean_agent = df.copy()
local_scope = {'df': df_clean_agent, 'pd': pd}

exec(cleaning_code, {}, local_scope)

df_final = local_scope['df']
df_final.head()

In [ ]:
# Après vérification manuelle du script généré : exécution dans un espace de noms local.
df_clean_agent_restricted = df.copy()
local_scope_restricted = {'df': df_clean_agent_restricted, 'pd': pd}

exec(cleaning_code_restricted, {}, local_scope_restricted)

df_final_restricted = local_scope_restricted['df']
df_final_restricted.head()

## 5. Le Grand Final : Tout en une fois !
Voici le script complet qui enchaîne tout : Analyse Robuste -> Génération Code -> Exécution.

In [ ]:
# === 1. ANALYSE ROBUSTE ===

# Prompt optimisé pour pousser l'IA dans ses retranchements
prompt_robust_audit = f"""
Tu es un Expert Data Engineer impitoyable sur la qualité de données.
Audite ce dataset pour produire un plan de nettoyage en béton.

DATA HEAD:
{head_str}

DATA INFO:
{info_str}

Instructions de nettoyage :
1. COLONNES : Le dataset contient Transaction_ID, Date, Client_Name, Product_Category, Unit_Price et Quantity.
2. STANDARDISATION : Repère les variantes de casse et les fautes manifestes de catégories sans modifier arbitrairement les noms de clients.
3. PRIX : Repère les symboles ou suffixes de devise dans Unit_Price et propose une conversion numérique robuste.
4. LOGIQUE MÉTIER : Vérifie les identifiants répétés et les quantités négatives. Ne fixe pas de seuil d'anomalie non justifié.
5. DATES : Prends en compte les formats ISO, jour/mois/année et les valeurs non datées comme 'TBD'.

Retourne une liste structured (JSON) des actions à mener.
"""

print("🔍 Analyse approfondie en cours...")
response_audit = generate_content(prompt_robust_audit, response_schema=DataAudit)
robust_issues = json.loads(response_audit.text)['issues']
print(f" {len(robust_issues)} problèmes détectés.")


# === 2. GENERATION CODE ===
print(" Génération du code de réparation...")
robust_code = generate_cleaning_code(df, robust_issues)
print("Code généré :")
print(robust_code)


# === 3. EXECUTION ===
print(" Exécution...")
df_clean_final = df.copy()
scope = {'df': df_clean_final, 'pd': pd}
exec(robust_code, {}, scope)
df_final_result = scope['df']
print("Terminé ! Aperçu avant/après :")
display(df.head(3))
display(df_final_result.head(3))
df_final_result.info()

## 6. Préparer le dépôt Git

La racine du dépôt est le dossier qui contient les notebooks numérotés (`01_manual_cleaning.ipynb`, `02_agent_logic.ipynb`), `dirty_sales_data.csv`, `llm_utils.py`, `requirements.txt` et `setup_guide.txt`. Les fichiers utilisent des noms en minuscules, sans accents ni espaces, séparés par des underscores (`snake_case`). Gardez les chemins relatifs afin que les notebooks et les scripts fonctionnent après un clone.

Le fichier `.gitignore` du projet exclut les environnements virtuels, les caches et `.env`. Seul `.env.example`, qui ne contient aucune vraie clé, doit être partagé. Avant le premier commit, vérifiez la liste des fichiers :

```powershell
git init
git status --short
git add .
git status
git commit -m "Initialise le projet de nettoyage de données"
```

Ne validez pas de clé API, de dossier `.venv` ni de sortie de notebook contenant des données sensibles.